# Linearisierung einer Funktion (Taylorentwicklung 1. Ordnung)

Dieses Notebook stellt eine Funktion `linearisieren()` bereit, die eine (ggf. mehrdimensionale) Funktion symbolisch um einen Arbeitspunkt linearisiert und dabei **alle Zwischenschritte** ausgibt:

1. Ausgangsfunktion
2. Partielle Ableitungen (Jacobi-Matrix)
3. Auswertung der Ableitungen am Arbeitspunkt
4. Auswertung der Funktion selbst am Arbeitspunkt
5. Zusammensetzen der linearisierten (linearen) Näherung
6. Optional: vereinfachte / ausmultiplizierte Form

Funktioniert sowohl für **eine** Variable als auch für **mehrere** Variablen (z.B. Zustandsgrößen in der Regelungstechnik).

In [6]:
import sympy as sp
from sympy import Symbol, Function, Matrix, Eq, latex
from IPython.display import display, Markdown

sp.init_printing()

## Die Linearisierungsfunktion

In [7]:
def linearisieren(f, variablen, arbeitspunkt, funktionsname="f", zeige_schritte=True):
    """
    Linearisiert eine (skalare) Funktion f um einen Arbeitspunkt (Taylorreihe 1. Ordnung).

    Parameter
    ---------
    f : sympy-Ausdruck
        Die zu linearisierende Funktion, z.B. f = x**2 * sp.sin(y)
    variablen : list[sympy.Symbol]
        Liste der Variablen, nach denen abgeleitet wird, z.B. [x, y]
    arbeitspunkt : dict[sympy.Symbol, float/int/sympy]
        Arbeitspunkt, z.B. {x: 1, y: 0}
    funktionsname : str
        Nur für die Ausgabe (z.B. 'f' oder 'F')
    zeige_schritte : bool
        Wenn True, werden alle Zwischenschritte per Markdown/Display ausgegeben

    Rückgabe
    --------
    dict mit den Zwischenergebnissen:
        'f0'            : Funktionswert am Arbeitspunkt
        'ableitungen'   : dict Variable -> abgeleiteter Ausdruck (unausgewertet)
        'ableitungen_ap': dict Variable -> Ableitung ausgewertet am Arbeitspunkt
        'f_lin'         : linearisierte Funktion (als sympy-Ausdruck, mit Delta-Termen)
        'f_lin_vereinfacht' : ausmultiplizierte/vereinfachte Form in den Originalvariablen
    """

    def zeige(md_text=None, ausdruck=None):
        if not zeige_schritte:
            return
        if md_text is not None:
            display(Markdown(md_text))
        if ausdruck is not None:
            display(ausdruck)

    # --- Schritt 1: Ausgangsfunktion ---
    zeige(f"### Schritt 1: Ausgangsfunktion\n$${funktionsname}({', '.join(str(v) for v in variablen)}) = {latex(f)}$$")

    # --- Schritt 2: Partielle Ableitungen ---
    zeige("### Schritt 2: Partielle Ableitungen")
    ableitungen = {}
    for v in variablen:
        d = sp.diff(f, v)
        ableitungen[v] = d
        zeige(f"$$\\dfrac{{\\partial {funktionsname}}}{{\\partial {latex(v)}}} = {latex(d)}$$")

    # --- Schritt 3: Arbeitspunkt anzeigen ---
    ap_str = ", \\; ".join(f"{latex(v)} = {latex(sp.nsimplify(w))}" for v, w in arbeitspunkt.items())
    zeige(f"### Schritt 3: Arbeitspunkt\n$${ap_str}$$")

    # --- Schritt 4: Funktionswert am Arbeitspunkt ---
    f0 = f.subs(arbeitspunkt)
    zeige(f"### Schritt 4: Funktionswert am Arbeitspunkt\n$${funktionsname}_0 = {funktionsname}(\\text{{AP}}) = {latex(f0)}$$")

    # --- Schritt 5: Ableitungen am Arbeitspunkt auswerten ---
    zeige("### Schritt 5: Ableitungen am Arbeitspunkt auswerten")
    ableitungen_ap = {}
    for v in variablen:
        wert = ableitungen[v].subs(arbeitspunkt)
        ableitungen_ap[v] = wert
        zeige(f"$$\\left.\\dfrac{{\\partial {funktionsname}}}{{\\partial {latex(v)}}}\\right|_{{\\text{{AP}}}} = {latex(wert)}$$")

    # --- Schritt 6: Linearisierte Funktion aufstellen (mit Delta-Termen) ---
    delta_terme = []
    delta_symbole = {}
    for v in variablen:
        dv = Symbol(f"\\Delta {sp.latex(v)}")
        delta_symbole[v] = dv
        delta_terme.append(ableitungen_ap[v] * dv)

    f_lin_delta = f0 + sum(delta_terme)
    zeige("### Schritt 6: Linearisierte Funktion (mit Δ-Größen)")
    summe_latex = " + ".join(
        f"{latex(ableitungen_ap[v])} \\, \\Delta {latex(v)}" for v in variablen
    )
    zeige(f"$${funktionsname}_{{\\text{{lin}}}} = {funktionsname}_0 + {summe_latex} = {latex(f0)} + {summe_latex}$$")

    # --- Schritt 7: Alternative Form direkt in den Originalvariablen ---
    # f_lin(x) = f0 + sum( df/dxi |AP * (xi - xi0) )
    terme_original = []
    for v in variablen:
        x0 = arbeitspunkt[v]
        terme_original.append(ableitungen_ap[v] * (v - x0))
    f_lin_original = sp.expand(f0 + sum(terme_original))

    zeige("### Schritt 7: Linearisierte Funktion in den Originalvariablen")
    summe_latex2 = " + ".join(
        f"{latex(ableitungen_ap[v])} \\,({latex(v)} - {latex(arbeitspunkt[v])})" for v in variablen
    )
    zeige(
        f"$${funktionsname}_{{\\text{{lin}}}}({', '.join(str(v) for v in variablen)}) "
        f"= {latex(f0)} + {summe_latex2} = {latex(f_lin_original)}$$"
    )

    return {
        "f0": f0,
        "ableitungen": ableitungen,
        "ableitungen_ap": ableitungen_ap,
        "f_lin": f_lin_delta,
        "f_lin_vereinfacht": f_lin_original,
    }

## Beispiel 1: Funktion einer Variable

$f(x) = x^2 \cdot \sin(x)$, Linearisierung um $x_0 = 1$

In [8]:
x = sp.symbols('x')
f1 = x**2 * sp.sin(x)

ergebnis1 = linearisieren(f1, [x], {x: 1}, funktionsname="f")

### Schritt 1: Ausgangsfunktion
$$f(x) = x^{2} \sin{\left(x \right)}$$

### Schritt 2: Partielle Ableitungen

$$\dfrac{\partial f}{\partial x} = x^{2} \cos{\left(x \right)} + 2 x \sin{\left(x \right)}$$

### Schritt 3: Arbeitspunkt
$$x = 1$$

### Schritt 4: Funktionswert am Arbeitspunkt
$$f_0 = f(\text{AP}) = \sin{\left(1 \right)}$$

...

$$\left.\dfrac{\partial f}{\partial x}\right|_{\text{AP}} = \cos{\left(1 \right)} + 2 \sin{\left(1 \right)}$$

### Schritt 6: Linearisierte Funktion (mit Δ-Größen)

$$f_{\text{lin}} = f_0 + \cos{\left(1 \right)} + 2 \sin{\left(1 \right)} \, \Delta x = \sin{\left(1 \right)} + \cos{\left(1 \right)} + 2 \sin{\left(1 \right)} \, \Delta x$$

### Schritt 7: Linearisierte Funktion in den Originalvariablen

$$f_{\text{lin}}(x) = \sin{\left(1 \right)} + \cos{\left(1 \right)} + 2 \sin{\left(1 \right)} \,(x - 1) = x \cos{\left(1 \right)} + 2 x \sin{\left(1 \right)} - \sin{\left(1 \right)} - \cos{\left(1 \right)}$$

## Beispiel 2: Funktion mehrerer Variablen

$g(x, y) = x^2 \cdot y + \cos(y)$, Linearisierung um $(x_0, y_0) = (2, 0)$

(z.B. relevant für die Linearisierung von Zustandsgleichungen $\dot{x} = g(x, u)$ in der Regelungstechnik)

In [9]:
x, y = sp.symbols('x y')
g = x**2 * y + sp.cos(y)

ergebnis2 = linearisieren(g, [x, y], {x: 2, y: 0}, funktionsname="g")

### Schritt 1: Ausgangsfunktion
$$g(x, y) = x^{2} y + \cos{\left(y \right)}$$

### Schritt 2: Partielle Ableitungen

$$\dfrac{\partial g}{\partial x} = 2 x y$$

$$\dfrac{\partial g}{\partial y} = x^{2} - \sin{\left(y \right)}$$

### Schritt 3: Arbeitspunkt
$$x = 2, \; y = 0$$

...

$$\left.\dfrac{\partial g}{\partial y}\right|_{\text{AP}} = 4$$

### Schritt 6: Linearisierte Funktion (mit Δ-Größen)

$$g_{\text{lin}} = g_0 + 0 \, \Delta x + 4 \, \Delta y = 1 + 0 \, \Delta x + 4 \, \Delta y$$

### Schritt 7: Linearisierte Funktion in den Originalvariablen

$$g_{\text{lin}}(x, y) = 1 + 0 \,(x - 2) + 4 \,(y - 0) = 4 y + 1$$

## Beispiel 3: Numerische Auswertung der Linearisierung

Die linearisierte Funktion (Ergebnis `f_lin_vereinfacht`) kann anschließend ganz normal mit `.subs()` numerisch ausgewertet werden, z.B. um zu prüfen, wie gut die Näherung nahe des Arbeitspunkts ist.

In [10]:
# Vergleich Original- vs. linearisierte Funktion in der Naehe des Arbeitspunkts (Beispiel 1)
x = sp.symbols('x')
for x_test in [0.8, 1.0, 1.2, 1.5]:
    original_wert = f1.subs(x, x_test)
    linear_wert = ergebnis1["f_lin_vereinfacht"].subs(x, x_test)
    print(f"x = {x_test:>4}: original = {float(original_wert):.5f}, "
          f"linearisiert = {float(linear_wert):.5f}, "
          f"Fehler = {float(original_wert - linear_wert):.5f}")

x =  0.8: original = 0.45911, linearisiert = 0.39682, Fehler = 0.06229
x =  1.0: original = 0.84147, linearisiert = 0.84147, Fehler = -0.00000
x =  1.2: original = 1.34214, linearisiert = 1.28612, Fehler = 0.05602
x =  1.5: original = 2.24436, linearisiert = 1.95309, Fehler = 0.29127


## Eigene Funktion linearisieren

Einfach in der Zelle unten `f`, `variablen` und den `arbeitspunkt` anpassen:

In [15]:
# --- Hier eigene Funktion eintragen ---
a, b = sp.symbols('x y')            # eigene Variablen definieren
eigene_funktion = sp.cos(x)   # eigene Funktion eintragen
eigener_arbeitspunkt = {a: 0, b: 0}      # Arbeitspunkt eintragen

ergebnis_eigen = linearisieren(
    eigene_funktion,
    [a, b],
    eigener_arbeitspunkt,
    funktionsname="h"
)

### Schritt 1: Ausgangsfunktion
$$h(x, y) = \cos{\left(x \right)}$$

### Schritt 2: Partielle Ableitungen

$$\dfrac{\partial h}{\partial x} = - \sin{\left(x \right)}$$

$$\dfrac{\partial h}{\partial y} = 0$$

### Schritt 3: Arbeitspunkt
$$x = 0, \; y = 0$$

...

$$\left.\dfrac{\partial h}{\partial y}\right|_{\text{AP}} = 0$$

### Schritt 6: Linearisierte Funktion (mit Δ-Größen)

$$h_{\text{lin}} = h_0 + 0 \, \Delta x + 0 \, \Delta y = 1 + 0 \, \Delta x + 0 \, \Delta y$$

### Schritt 7: Linearisierte Funktion in den Originalvariablen

$$h_{\text{lin}}(x, y) = 1 + 0 \,(x - 0) + 0 \,(y - 0) = 1$$

## Paket-Integration (Laplace + Reglerentwurf)

Zusätzlich zur Linearisierung binden wir hier die Kernfunktionen aus dem Paket ein.

- Eingang: Ausdruck oder Streckenparameter
- Ausgabe: Laplace-Ergebnisse und Reglerparameter

In [ ]:
import sympy as sp
from regelungstechnik import (
    laplace_transform,
    inverse_laplace,
    partialbruchzerlegung,
    reglerparameter_nach_verfahren,
    phasenkorrekturglied_auslegung,
)

# Eingabe (Laplace)
t = sp.symbols('t', positive=True)
f_t = t * sp.exp(-2 * t)
print("EINGABE f(t) =", f_t)

# Ausgabe (Laplace + inverse Laplace)
res_L = laplace_transform(f_t)
print("\nAUSGABE F(s) =", res_L["ergebnis"])

res_iL = inverse_laplace(res_L["ergebnis"])
print("AUSGABE f(t) (Ruecktransformation) =", res_iL["ergebnis"])

# Ausgabe (Partialbruch)
num_pb = [1, 3]
den_pb = [1, 4, 3]
res_pb = partialbruchzerlegung(num_pb, den_pb)
print("\nEINGABE Partialbruch: num=", num_pb, " den=", den_pb)
print("AUSGABE Partialbruch:", res_pb["ergebnis"])

# Ausgabe (Reglerparameter)
res_regler = reglerparameter_nach_verfahren(
    reglertyp="PI",
    verfahren="ziegler-nichols",
    modus="offen",
    K=2.0,
    T=5.0,
    K_T=1.0,
)
print("\nAUSGABE Reglerparameter (PI, ZN offen):")
print(res_regler["ergebnis"])

# Ausgabe (Phasenkorrekturglied)
res_pk = phasenkorrekturglied_auslegung(
    typ="anhebend",
    phi_grad=35,
    omega_c=2.0,
    K=1.0,
)
print("\nAUSGABE Phasenkorrekturglied:")
print(res_pk["ergebnis"])